In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ['LANGSMITH_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

In [3]:
os.environ["LANGSMITH_TEST_PROJECT"] = "TestProject"

In [4]:
from langchain_core.tools import tool
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph.message import add_messages, BaseMessage
from langchain_groq import ChatGroq

In [5]:
from langchain_tavily import TavilySearch

In [6]:
llm = ChatGroq(model = "openai/gpt-oss-20b", groq_api_key = groq_api_key)

In [7]:
class State(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [8]:
@tool
def add(a:int, b:int):
    """Adding numbers"""
    return a+b

@tool
def weather(location:str):
    """Weather of a city"""
    return f"Its always cold in {location}."

tavily = TavilySearch(max_results = 2)

tools = [add, weather, tavily]
llm_with_tools = llm.bind_tools(tools)

#Node Specialization
def llm_chatbot(state:State):
    return {"messages":[llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)


In [9]:
#nodes
builder.add_node("tool_calling_llm",llm_chatbot)
builder.add_node("tools",ToolNode(tools))

#edges
builder.add_edge(START,"tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm",tools_condition)
builder.add_edge("tools","tool_calling_llm")

graph=builder.compile()

In [10]:
graph.invoke({"messages":"what is the weather in NYC and add 5 plus 8"})

{'messages': [HumanMessage(content='what is the weather in NYC and add 5 plus 8', additional_kwargs={}, response_metadata={}, id='36381288-954a-4791-ad13-d1c3de0a26a5'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants weather in NYC and addition 5+8. Need to use functions: weather and add. Use weather with location "NYC" or "New York City". Then add a=5, b=8. Let\'s do that.', 'tool_calls': [{'id': 'fc_fec54bc0-82c0-4d2c-b2e3-908af91cc81d', 'function': {'arguments': '{"location":"New York City"}', 'name': 'weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 366, 'total_tokens': 440, 'completion_time': 0.095945862, 'completion_tokens_details': {'reasoning_tokens': 50}, 'prompt_time': 0.017761485, 'prompt_tokens_details': None, 'queue_time': 0.169021548, 'total_time': 0.113707347}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9b8528b477', 'service_tier': 'on_demand', 'finish_reason': '

In [34]:
response = graph.invoke({"messages":"Hey what is 8 plus 98?"})

In [36]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Hey what is 8 plus 98?
================================== Ai Message ==================================
Tool Calls:
  add (fc_6d2b092e-2115-4748-a884-437cbbcb8983)
 Call ID: fc_6d2b092e-2115-4748-a884-437cbbcb8983
  Args:
    a: 8
    b: 98
================================= Tool Message =================================
Name: add

106
================================== Ai Message ==================================

106


In [40]:
response = graph.invoke({"messages":"Hey What is the latest news of Korutla, telanagana, India, and whats the weather in San Jose and also add two numbers of 65 plus 79? "})

In [42]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Hey What is the latest news of Korutla, telanagana, India, and whats the weather in San Jose and also add two numbers of 65 plus 79? 
================================== Ai Message ==================================
Tool Calls:
  tavily_search (fc_a617260e-c2ff-4928-addb-796e601c74f8)
 Call ID: fc_a617260e-c2ff-4928-addb-796e601c74f8
  Args:
    query: Korutla Telangana news
    search_depth: basic
================================= Tool Message =================================
Name: tavily_search

{"query": "Korutla Telangana news", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://telanganatoday.com/tag/korutla", "title": "Korutla Latest News, Korutla News", "content": "Man injured in Korutla stabbing dies during treatment in hospital. Satyanarayana (49), who was stabbed a few days ago in Korutla, died during treatment at a ...Read more", "score": 0.6453435, "raw_con